# FT 3B Solo — PIQA Direct Answer Fine-Tuning

**Purpose:** Fine-tune Qwen2.5-3B-Instruct to answer PIQA binary-choice physical reasoning questions **directly** (no guide, no pipeline). Evaluate on N=500 at seed=42 — the same split used in the paper.

PIQA format: each question has two solutions (sol1, sol2). Answer is `sol1` or `sol2`.
Random chance = 50%.

| Condition | Compute | Paper result |
|---|---|---|
| Baseline (1.5B×5) | 7.5B pp | 54.2% — 93.4% A-bias |
| CoT (1.5B×5) | 7.5B pp | 54.8% — debiased but not accurate |
| Base-3B ablation | 10.5B pp | 77.4% — Layer 1 sufficient |
| **FT 3B Solo (this run)** | **3.0B pp** | **TBD** |
| Guided pipeline | 10.5B pp | 78.4% |

> Seed=42 · N=500 · Same question split as original paper run

**Key question:** does fine-tuning the 3B model to answer PIQA directly reach the same accuracy as the guided pipeline (78.4%) at 3.5× less compute?

In [1]:
# CELL 1 — Install
!pip install trl
print("Done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 12.6 MB/s eta 0:00:0000:01
Done.


In [2]:
# CELL 2 — Login
from huggingface_hub import login
login("")  # paste your HF token
print("Login done")

Login done


In [3]:
# CELL 3 — Imports
import os, json, re, random, time
import torch
from collections import Counter
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer
from tqdm.notebook import tqdm

OUTPUT_DIR = "/content/piqa_ft3b_solo"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# CELL 4 — Config
# eval_seed=42, eval_n=500 MUST match paper exactly.
CONFIG = {
    "model_name"        : "Qwen/Qwen2.5-3B-Instruct",
    "dataset_name"      : "nthngdy/piqa",
    "eval_split"        : "validation",
    "train_split"       : "train",
    "eval_seed"         : 42,
    "eval_n"            : 500,
    "max_train_samples" : 1000,   # PIQA train ~16113 examples
    "lora_r"            : 16,
    "lora_alpha"        : 32,
    "lora_dropout"      : 0.05,
    "learning_rate"     : 2e-4,
    "num_epochs"        : 3,
    "batch_size"        : 4,
    "grad_accum"        : 4,
    "max_seq_length"    : 192,
    "max_new_tokens"    : 8,      # just a label — very short
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"        : 50,
}
print("Config ready. eval_seed=42, eval_n=500.")

Config ready. eval_seed=42, eval_n=500.


In [5]:
# CELL 5 — Load PIQA
# PIQA format: goal (question), sol1, sol2, label (0=sol1, 1=sol2)
print("Loading PIQA...")
raw_ds = load_dataset(CONFIG["dataset_name"])

def normalise_piqa(item):
    goal = str(item["goal"]).strip()
    sol1 = str(item["sol1"]).strip()
    sol2 = str(item["sol2"]).strip()
    label = int(item["label"])   # 0 = sol1 correct, 1 = sol2 correct
    # Build question with both options labelled A / B
    full_q = (
        f"Goal: {goal}\n\n"
        f"A. {sol1}\n"
        f"B. {sol2}"
    )
    answer = "A" if label == 0 else "B"
    return {
        "question" : full_q,
        "answer"   : answer,
        "sol1"     : sol1,
        "sol2"     : sol2,
        "label"    : label,
    }

eval_pool  = [normalise_piqa(x) for x in raw_ds[CONFIG["eval_split"]]]
train_pool = [normalise_piqa(x) for x in raw_ds[CONFIG["train_split"]]]

print(f"Validation pool : {len(eval_pool)}")
print(f"Train pool      : {len(train_pool)}")
ad = Counter(x["answer"] for x in eval_pool)
print(f"Answer dist (eval): {dict(ad)}  (expected ~50/50)")

Loading PIQA...


README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.66M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/502k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/301k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16113 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3084 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1838 [00:00<?, ? examples/s]

Validation pool : 1838
Train pool      : 16113
Answer dist (eval): {'A': 910, 'B': 928}  (expected ~50/50)


In [6]:
# CELL 6 — Eval / train split
# Eval: N=500 from validation with seed=42 — matches paper
random.seed(CONFIG["eval_seed"])
if CONFIG["eval_n"] < len(eval_pool):
    eval_data = random.sample(eval_pool, CONFIG["eval_n"])
else:
    eval_data = eval_pool

eval_questions = set(x["question"] for x in eval_data)

# Train: from train split (entirely separate from eval)
train_candidates = [x for x in train_pool
                    if x["question"] not in eval_questions]
random.seed(0)
if len(train_candidates) > CONFIG["max_train_samples"]:
    train_data = random.sample(train_candidates, CONFIG["max_train_samples"])
else:
    train_data = train_candidates

overlap = eval_questions & set(x["question"] for x in train_data)
print(f"Eval  : {len(eval_data)} questions (seed=42)")
print(f"Train : {len(train_data)} questions")
print(f"Overlap: {len(overlap)} (must be 0)")

ed = Counter(x["answer"] for x in eval_data)
print(f"Eval answer dist: {dict(ed)}")

Eval  : 500 questions (seed=42)
Train : 1000 questions
Overlap: 0 (must be 0)
Eval answer dist: {'A': 249, 'B': 251}


In [7]:
# CELL 7 — SFT prompt format
# Model trained to output only 'A' or 'B'.
# This directly measures whether fine-tuning can learn
# physical commonsense AND overcome the A-option collapse.

SYSTEM_PROMPT = (
    "You are a precise physical reasoning assistant.\n"
    "Read the goal and both solutions carefully.\n"
    "Respond with only A or B — the letter of the better solution."
)

def format_sft(item, tokenizer):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": item["question"]},
        {"role": "assistant", "content": item["answer"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

print("SFT format ready.")
print(f"Example:")
print(f"  {train_data[0]["question"]}")
print(f"  Answer: {train_data[0]["answer"]}")

SFT format ready.
Example:
  Goal: To make your hair grow thicker and faster,

A. wash your hair with biotin shampoo or take a biotin supplement every day.
B. wash your hair with biotin shampoo or take a biotin shower every day.
  Answer: A


In [8]:
# CELL 8 — Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer ready.")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.


In [9]:
# CELL 9 — Prepare HF Dataset
train_formatted = [format_sft(x, tokenizer) for x in train_data]
hf_train = Dataset.from_list(train_formatted)
print(f"Training examples: {len(hf_train)}")
print(f"Sample (first 280 chars):\n{hf_train[0]["text"][:280]}")

Training examples: 1000
Sample (first 280 chars):
<|im_start|>system
You are a precise physical reasoning assistant.
Read the goal and both solutions carefully.
Respond with only A or B — the letter of the better solution.<|im_end|>
<|im_start|>user
Goal: To make your hair grow thicker and faster,

A. wash your hair with biotin 


In [10]:
# CELL 10 — Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
)
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
base_model.enable_input_require_grads()
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

VRAM: 3.09GB


In [11]:
# CELL 11 — Attach LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    bias="none",
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [12]:
# CELL 12 — Fine-tune
# ~50-70 min on T4 with 14000 training examples
training_args = TrainingArguments(
    output_dir=f"{OUTPUT_DIR}/checkpoints",
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=1,        # ← down from 4
    gradient_accumulation_steps=16,       # ← up from 4 (keeps effective batch=16)
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="adafactor",                    # ← saves ~2GB vs Adam
    logging_steps=100,
    save_strategy="epoch",
    report_to="none",
    gradient_checkpointing=True,          # ← enable here too
    dataloader_num_workers=0,
    seed=42,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [13]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


trainer = SFTTrainer(
    model=model, train_dataset=hf_train,
    processing_class=tokenizer, args=training_args,
)
t0 = time.time()
trainer.train()
print(f"Training done in {(time.time()-t0)/60:.1f} min.")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
100,1.033706


Training done in 31.4 min.


In [14]:
# CELL 13 — Save model
ft_model_path = f"{OUTPUT_DIR}/ft_model"
trainer.save_model(ft_model_path)
tokenizer.save_pretrained(ft_model_path)
print(f"Saved to {ft_model_path}")

Saved to /content/piqa_ft3b_solo/ft_model


In [15]:
# CELL 14 — Answer extraction (A or B)
def extract_ab_answer(text):
    text = text.strip()
    # 1. Single letter on first line
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        m = re.match(r"^\**([AB])\**[.):,]?$", lines[0], re.IGNORECASE)
        if m: return m.group(1).upper()
    # 2. 'answer is X' or 'solution X'
    m = re.search(r"(?:answer\s+is|solution\s+is|answer:|correct)\s*\**([AB])\**",
                  text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # 3. Bold letter
    m = re.search(r"\*\*([AB])\*\*", text)
    if m: return m.group(1).upper()
    # 4. Parenthesised
    m = re.search(r"\(([AB])\)", text)
    if m: return m.group(1).upper()
    # 5. Any standalone A or B
    m = re.search(r"\b([AB])\b", text)
    if m: return m.group(1).upper()
    return ""

# Tests
_t = ["A","**B**","answer is A","Solution B is better","(A)"]
_e = ["A","B","A","B","A"]
ok = all(extract_ab_answer(t)==e for t,e in zip(_t,_e))
print("Extractor:", "PASSED" if ok else "FAIL")

Extractor: PASSED


In [16]:
# CELL 15 — Reload model for eval
del model, base_model, trainer
torch.cuda.empty_cache()

eval_base = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], torch_dtype=torch.float16, device_map="auto"
).eval()
ft_model = PeftModel.from_pretrained(eval_base, ft_model_path).eval()
print("Fine-tuned model loaded for eval.")
print(f"VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Fine-tuned model loaded for eval.
VRAM: 6.31GB


In [17]:
# CELL 16 — Eval function (single greedy pass = 3.0B pp)
EVAL_SYSTEM = (
    "You are a precise physical reasoning assistant.\n"
    "Read the goal and both solutions carefully.\n"
    "Respond with only A or B — the letter of the better solution."
)

def run_ft_solo(question):
    messages = [
        {"role": "system", "content": EVAL_SYSTEM},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=512)
    device = next(ft_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_toks, skip_special_tokens=True).strip()

# Verification: 20 questions
v_correct = 0; v_empty = 0
for item in eval_data[:20]:
    raw  = run_ft_solo(item["question"])
    pred = extract_ab_answer(raw)
    if not pred: v_empty += 1
    if pred == item["answer"]: v_correct += 1
print(f"Verification (20 q): {v_correct}/20 = {v_correct/20*100:.0f}%")
print(f"Empty: {v_empty}/20")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Verification (20 q): 17/20 = 85%
Empty: 0/20


In [18]:
# CELL 17 — Full evaluation N=500
# ~5-8 min on T4
print(f"Evaluating {CONFIG['eval_n']} questions | compute: 3.0B pp per question")
print("-"*60)

results = []; start_idx = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f: ck = json.load(f)
    start_idx = ck.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed from {start_idx}")

t0 = time.time()
for idx in tqdm(range(start_idx, len(eval_data)), desc="PIQA-FT3B"):
    item = eval_data[idx]
    try:
        raw  = run_ft_solo(item["question"])
        pred = extract_ab_answer(raw)
        gt   = item["answer"]
        results.append({
            "idx"          : idx,
            "question"     : item["question"],
            "gt_answer"    : gt,
            "raw_output"   : raw,
            "final_answer" : pred,
            "correct"      : (pred == gt),
            "empty"        : (pred == ""),
        })
    except Exception as e:
        results.append({
            "idx": idx, "question": item["question"],
            "gt_answer": item["answer"], "raw_output": "",
            "final_answer": "", "correct": False,
            "empty": True, "error": str(e)
        })

    if (idx+1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in results: f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx+1}, f)
        acc  = sum(r["correct"] for r in results)/len(results)*100
        mins = (time.time()-t0)/60
        print(f"  [{idx+1:4d}] acc={acc:.1f}%  ({mins:.1f}min)")

with open(CONFIG["results_file"], "w") as f:
    for r in results: f.write(json.dumps(r) + "\n")

n_correct = sum(r["correct"] for r in results)
n_empty   = sum(r["empty"]   for r in results)
print(f"\nAccuracy : {n_correct}/{len(results)} = {n_correct/len(results)*100:.1f}%")
print(f"Empty    : {n_empty}")

Evaluating 500 questions | compute: 3.0B pp per question
------------------------------------------------------------


PIQA-FT3B:   0%|          | 0/500 [00:00<?, ?it/s]

  [  50] acc=86.0%  (0.2min)
  [ 100] acc=86.0%  (0.4min)
  [ 150] acc=85.3%  (0.6min)
  [ 200] acc=82.0%  (0.8min)
  [ 250] acc=82.0%  (1.0min)
  [ 300] acc=83.3%  (1.2min)
  [ 350] acc=84.0%  (1.4min)
  [ 400] acc=83.8%  (1.7min)
  [ 450] acc=83.6%  (1.9min)
  [ 500] acc=83.4%  (2.1min)

Accuracy : 417/500 = 83.4%
Empty    : 0


In [19]:
# CELL 18 — Position bias check (most important for PIQA)
# Baseline had 93.4% A-selection — does FT Solo fix this?
pred_dist = Counter(r["final_answer"] for r in results)
gt_dist   = Counter(r["gt_answer"]    for r in results)

print("Option selection rates:")
print(f"  {"Condition":<25} {"A %":>8} {"B %":>8}")
print("-"*44)
a_rate_ft  = pred_dist.get("A",0) / len(results) * 100
b_rate_ft  = pred_dist.get("B",0) / len(results) * 100
a_rate_gt  = gt_dist.get("A",0)   / len(results) * 100
print(f"  {"Baseline (paper)":<25} {93.4:>7.1f}% {6.6:>7.1f}%  ← collapsed")
print(f"  {"CoT (paper)":<25} {44.2:>7.1f}% {55.8:>7.1f}%  ← debiased")
print(f"  {"Base-3B ablation (paper)":<25} {46.0:>7.1f}% {54.0:>7.1f}%  ← debiased")
print(f"  {"FT 3B Solo (this run)":<25} {a_rate_ft:>7.1f}% {b_rate_ft:>7.1f}%")
print(f"  {"Ground truth labels":<25} {a_rate_gt:>7.1f}% {100-a_rate_gt:>7.1f}%")
print()
if a_rate_ft <= 55.0:
    print("  ✓ A-option bias suppressed by fine-tuning")
else:
    print(f"  ✗ A-option bias persists: {a_rate_ft:.1f}% A selections")

Option selection rates:
  Condition                      A %      B %
--------------------------------------------
  Baseline (paper)             93.4%     6.6%  ← collapsed
  CoT (paper)                  44.2%    55.8%  ← debiased
  Base-3B ablation (paper)     46.0%    54.0%  ← debiased
  FT 3B Solo (this run)        44.8%    55.2%
  Ground truth labels          49.8%    50.2%

  ✓ A-option bias suppressed by fine-tuning


In [20]:
# CELL 19 — Final comparison table
ft_acc = sum(r["correct"] for r in results) / len(results) * 100

# Paper confirmed values for PIQA
BASELINE  = 54.2
COT       = 54.8
BASE_3B   = 77.4   # untuned 3B ablation (from results__6_.jsonl)
GUIDED    = 78.4

print("="*65)
print("PIQA — FULL COMPUTE-ACCURACY COMPARISON")
print("="*65)
print(f"  Condition              | Compute   | Accuracy | A-bias")
print(f"  -----------------------|-----------|----------|-------")
print(f"  Baseline (1.5B×5)      | 7.5B pp   | {BASELINE}%  | 93.4%")
print(f"  CoT (1.5B×5)           | 7.5B pp   | {COT}%  | 44.2%")
print(f"  FT 3B Solo (this run)  | 3.0B pp   | {ft_acc:.1f}%  | {a_rate_ft:.1f}%")
print(f"  Base-3B ablation       | 10.5B pp  | {BASE_3B}%  | 46.0%")
print(f"  Guided pipeline        | 10.5B pp  | {GUIDED}%  | 42.1%")
print()
gap_vs_guided = GUIDED  - ft_acc
gap_vs_base   = ft_acc  - BASELINE
gap_vs_base3b = BASE_3B - ft_acc
print(f"  FT Solo vs Baseline    : {gap_vs_base:+.1f} pts  (at 2.5x less compute)")
print(f"  FT Solo vs Base-3B abl.: {-gap_vs_base3b:+.1f} pts  (same compute 10.5B vs 3.0B)")
print(f"  Guided vs FT Solo      : {gap_vs_guided:+.1f} pts  (guided costs +7.5B pp)")
print()
if ft_acc >= 77.0:
    print("  VERDICT: FT Solo matches guided pipeline.")
    print("  Fine-tuning alone achieves the result — pipeline not justified on PIQA.")
elif ft_acc >= 70.0:
    print(f"  VERDICT: FT Solo strong but guided adds {gap_vs_guided:.1f} pts at +7.5B pp.")
elif ft_acc >= 60.0:
    print(f"  VERDICT: FT Solo decent. Guided adds {gap_vs_guided:.1f} pts — pipeline justified.")
else:
    print(f"  VERDICT: FT Solo weak on PIQA. Guided adds {gap_vs_guided:.1f} pts.")
    print("  Fine-tuning to answer PIQA directly is insufficient — pipeline essential.")

PIQA — FULL COMPUTE-ACCURACY COMPARISON
  Condition              | Compute   | Accuracy | A-bias
  -----------------------|-----------|----------|-------
  Baseline (1.5B×5)      | 7.5B pp   | 54.2%  | 93.4%
  CoT (1.5B×5)           | 7.5B pp   | 54.8%  | 44.2%
  FT 3B Solo (this run)  | 3.0B pp   | 83.4%  | 44.8%
  Base-3B ablation       | 10.5B pp  | 77.4%  | 46.0%
  Guided pipeline        | 10.5B pp  | 78.4%  | 42.1%

  FT Solo vs Baseline    : +29.2 pts  (at 2.5x less compute)
  FT Solo vs Base-3B abl.: +6.0 pts  (same compute 10.5B vs 3.0B)
  Guided vs FT Solo      : -5.0 pts  (guided costs +7.5B pp)

  VERDICT: FT Solo matches guided pipeline.
  Fine-tuning alone achieves the result — pipeline not justified on PIQA.
